# HLS Video Player — Google Colab 起動ノート

このノートブックは hls-video-player を Colab 上で動作させます。**コードベースは Google Drive から取得**、メディア（ソース動画と変換出力）も Drive に永続化します。

## 事前準備（初回のみ）

Google Drive に次のディレクトリレイアウトでコードを配置してください（ローカルの `hls-video-player/` ディレクトリをそのままアップロード）:

```
MyDrive/
└── hls-video-player/
    ├── pyproject.toml
    ├── app/
    ├── hls_video/
    ├── static/
    └── media/
        ├── source/   ← 変換元 MP4 を置く（永続）
        ├── hls/      ← 変換出力（永続、再生成可能）
        └── sprites/  ← スプライト + VTT（永続、再生成可能）
```

コード更新は手元で編集して Drive に上書き同期するだけで済みます。

## 実行手順

1. FFmpeg + Python 依存インストール
2. Drive マウント
3. Drive → Colab ローカルへコード複製（I/O 高速化のため）
4. MEDIA_ROOT を Drive に設定
5. アプリ起動

## 1. FFmpeg + Python 依存

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null
!ffmpeg -version | head -1
!pip install -q gradio fastapi uvicorn python-multipart

## 2. Google Drive マウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Drive からコードを複製 + インストール

Drive 上のコードは直接実行せず、Colab ローカル (`/content/hls-video-player`) にコピーして使います。
Drive は FUSE 経由で I/O が遅く、インポートや FastAPI リロードが重くなるため。

メディア（`media/`）だけは Drive に残して永続化します（後のセルで MEDIA_ROOT を Drive 側に向ける）。

In [ ]:
import os, shutil

DRIVE_ROOT = '/content/drive/MyDrive/hls-video-player'
LOCAL_ROOT = '/content/hls-video-player'

assert os.path.isdir(DRIVE_ROOT), (
    f'{DRIVE_ROOT} が見つかりません。MyDrive 直下に hls-video-player/ を配置してください。'
)

# コードを rsync 相当でコピー。media/ は Drive に残すため除外。
if os.path.exists(LOCAL_ROOT):
    shutil.rmtree(LOCAL_ROOT)
shutil.copytree(
    DRIVE_ROOT,
    LOCAL_ROOT,
    ignore=shutil.ignore_patterns(
        'media', '__pycache__', '.pytest_cache', '.venv', '*.egg-info',
        '.git', '.claude', 'node_modules',
    ),
)

%cd {LOCAL_ROOT}
!ls
!pip install -q -e .

## 4. MEDIA_ROOT を Drive に設定

`media/` は Drive 上のものを直接マウントして使います（ソース動画のアップロードと変換結果がセッションを跨いで永続化）。

In [ ]:
import os

MEDIA = f'{DRIVE_ROOT}/media'
for sub in ('source', 'hls', 'sprites'):
    os.makedirs(f'{MEDIA}/{sub}', exist_ok=True)

os.environ['MEDIA_ROOT'] = MEDIA
os.environ['MAX_CONCURRENT_JOBS'] = '1'      # Colab は通常 2 vCPU
os.environ['FFMPEG_THREADS'] = '2'
os.environ['FFMPEG_PRESET'] = 'veryfast'
os.environ['GRADIO_SHARE'] = 'true'
os.environ['PORT'] = '7860'

!ls {MEDIA}

## 5. 起動

Gradio の公開 URL は出力された `*.gradio.live` をクリック。
セッション切断後も Drive 上の変換結果は残るので、再接続時はこのノートを頭から実行し直せば復帰します。

In [ ]:
import gradio as gr
from fastapi import FastAPI

from app.gradio_ui import build_ui
from app.static_mount import mount_static
from app.player_embed import router as player_router
from hls_video.config import media_root

media = media_root()
fapi = FastAPI()
mount_static(fapi, media_root=str(media), static_dir=f'{LOCAL_ROOT}/static')
fapi.include_router(player_router)

demo = build_ui(media_dir=media)
demo.queue()
gr.mount_gradio_app(fapi, demo, path='/')

# demo.launch() は Gradio 単体にしか share=True が効かないため、
# FastAPI 全体を uvicorn で立ち上げつつ、別で demo.launch(share=True) により
# gradio のトンネルを張る。トンネルは root (/) にマウントされた demo に接続する。
import threading, uvicorn
threading.Thread(
    target=lambda: uvicorn.run(fapi, host='0.0.0.0', port=7860, log_level='warning'),
    daemon=True,
).start()

print('Local:  http://localhost:7860  (Colab 内部)')
demo.launch(share=True, server_name='0.0.0.0', server_port=7861, prevent_thread_lock=False)

## コード更新時

手元で編集 → Drive に上書き → このノートを先頭から再実行、の流れで反映されます。
セル 3 (Drive → ローカル複製) がコピーし直しているので、ローカル側に残った古いコードは上書きされます。